# Audio experiment EDA
Explore saved reports; production transforms and training stay in src/. Synthetic tones do not establish real-world call recognition.

In [ ]:
import json
from pathlib import Path
report = json.loads(Path('../docs/esc10-results.json').read_text())
list(zip(report['classes'], report['test']['per_class_f1'], report['test']['positive_support']))

## Natural MAESTRO and pretrained transfer
Read saved aggregate evidence only. Existing holdout is observed; context and head are chosen by validation AP. Unsupported and rare classes remain explicit.

In [ ]:
baseline = json.loads(Path("../docs/maestro-results.json").read_text(encoding="utf-8"))["cloud_final_report"]
baseline["test_validation_thresholds"]["per_class"]

In [ ]:
transfer = json.loads(Path("../docs/transfer-results.json").read_text(encoding="utf-8"))
comparison = []
for context, result in transfer["contexts"].items():
    observed = result["observed_holdout"]["test_validation_thresholds"]
    comparison.append({"context": context, "validation_ap": result["validation_macro_ap"], "observed_holdout_ap": observed["macro_ap_supported"], "all_observed_f1": observed["macro_f1_all_observed_classes"], "micro_f1": observed["micro_f1"]})
comparison

In [ ]:
selected = transfer["contexts"][transfer["selected_context_key"]]
selected["observed_holdout"]["test_validation_thresholds"]["per_class"]

## Controlled postprocessing and learned-head follow-up
Saved aggregates only. The observed test is retrospective; V6 keeps V5 after every new configuration/mixture fails validation. Training and inference remain in package modules.

In [ ]:
study = json.loads(Path("../docs/five-second-learned-v6-selection.json").read_text(encoding="utf-8"))
[(c["config"]["name"], c["equal_seed_ensemble_validation_ap"], c["individual_seed_std_ap"]) for c in study["candidates"]]

In [ ]:
post = json.loads(Path("../docs/postprocess-v5-observed-holdout.json").read_text(encoding="utf-8"))
before = {row["class"]: row for row in post["five_second_baseline"]["per_class"]}
[(row["class"], before[row["class"]]["ap"], row["ap"]) for row in post["selected"]["per_class"] if row["ap"] is not None and row["ap"] < before[row["class"]]["ap"]]